# Ollama Barrier Details Extraction
Standardized evaluation notebook comparing two approaches for spatial extraction using Ollama and Pydantic.

In [1]:
from ollama import chat
from pydantic import BaseModel, Field
from typing import List, Dict, Any

## 1. Evaluation Logic
Shared evaluation functions to score the models parsing lists directly from native list objects extracted.

In [2]:
def check_center_length(actual_list: list, expected_dict: dict) -> bool:
    return (actual_list[1] == expected_dict.get("center")) and \
           (actual_list[2] == expected_dict.get("side_lengths"))

def check_min_max(actual_list: list, expected_dict: dict) -> bool:
    return (actual_list[3] == expected_dict.get("min_point")) and \
           (actual_list[4] == expected_dict.get("max_point"))

def evaluate_test_case(test_id: int, actual_list: list) -> dict:
    prompt_text = prompts[test_id]
    expected_dict = expected_answers[test_id]
    
    expected_list = [
        expected_dict.get("shape_type"),
        expected_dict.get("center"),
        expected_dict.get("side_lengths"),
        expected_dict.get("min_point"),
        expected_dict.get("max_point")
    ]
    
    is_correct_full = (actual_list == expected_list)
    is_correct_cl = check_center_length(actual_list, expected_dict)
    is_correct_mm = check_min_max(actual_list, expected_dict)
    
    print(f"\n================ TEST CASE {test_id} ================")
    print(f"Question:    {prompt_text}")
    print("-" * 50)
    print(f"Ollama Ans:  {actual_list}")
    print(f"True Answer: {expected_list}")
    print("-" * 50)
    
    print(f"Full Match:          {'✅' if is_correct_full else '❌'}")
    print(f"Center/Length Match: {'✅' if is_correct_cl else '❌'}")
    print(f"Min/Max Match:       {'✅' if is_correct_mm else '❌'}")
    
    return {
        "full": is_correct_full,
        "cl": is_correct_cl,
        "mm": is_correct_mm
    }

## 2. Dataset
Defines the test prompts and their expected geometries.

In [3]:
prompts = [
    "A box is centred at (0, 0, 0) with sides 10 × 8 × 6. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box has its bottom-left-front corner at (−4, 1, 0) and its top-right-back corner at (8, 7, 10). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (5, 5, −5) with sides 14 × 10 × 4. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box runs from (−10, −5, −2) to (2, 5, 6). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (−3, −3, −3) with equal sides of length 12. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The region is a box centred at (8, 2, 6) with sides 6 × 16 × 8. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box with its near-bottom-left corner at (0, −8, 1) and its far-top-right corner at (10, 0, 9) defines the zone. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (−5, 10, 3) with sides 8 × 4 × 14. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box stretches from (−6, −6, 0) to (6, 6, 12). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (2, −7, 5) with sides 20 × 6 × 10. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box centred at (−2, 3, −5) has sides 16 × 4 × 12. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box extends from (1, −4, 2) to (13, 4, 10). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (3, 8, −4) with sides 10 × 6 × 8. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box runs from (−9, −3, −6) to (3, 9, 0). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (6, −9, 1) with sides 14 × 8 × 10. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box centred at (0, 4, −2) has sides 10 × 8 × 6. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box extends from (−5, 0, −5) to (5, 10, 5). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (9, −1, 4) with sides 6 × 12 × 8. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box centred at (−7, 0, 8) has sides 4 × 20 × 6. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box extends from (0, 0, 0) to (8, 16, 6). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box is centred at (4, −2, 7) with sides 10 × 6 × 8. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box runs from (−8, 1, −3) to (4, 9, 5). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (−1, 6, 2) with sides 12 × 8 × 10. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box centred at (10, 0, −4) has sides 8 × 14 × 6. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box extends from (2, −10, 0) to (14, −2, 8). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box is centred at (−6, −4, 9) with sides 6 × 10 × 4. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (0, −5, −8) with sides 20 × 10 × 6. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box runs from (−3, 3, 1) to (9, 11, 9). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (7, 3, −6) with sides 4 × 8 × 12. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box centred at (−4, 8, 0) has sides 16 × 6 × 10. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box extends from (−2, −2, −10) to (10, 6, 0). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box is centred at (5, −8, 3) with sides 10 × 4 × 14. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box runs from (0, −12, −4) to (16, 0, 8). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (−9, 2, 6) with sides 6 × 12 × 8. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box centred at (3, 0, −9) has sides 18 × 8 × 6. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box extends from (−11, −5, 2) to (1, 7, 10). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box is centred at (6, 6, −2) with sides 8 × 8 × 16. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box runs from (−4, 0, −7) to (8, 10, 5). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "The box is centred at (−5, −7, 4) with sides 14 × 6 × 10. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.",
    "A box centred at (1, −3, 11) has sides 12 × 10 × 8. Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point."
]

expected_answers = [
    {"shape_type": "cuboid", "center": [0.0, 0.0, 0.0], "side_lengths": [10.0, 8.0, 6.0], "min_point": [-5.0, -4.0, -3.0], "max_point": [5.0, 4.0, 3.0]},
    {"shape_type": "cuboid", "center": [2.0, 4.0, 5.0], "side_lengths": [12.0, 6.0, 10.0], "min_point": [-4.0, 1.0, 0.0], "max_point": [8.0, 7.0, 10.0]},
    {"shape_type": "cuboid", "center": [5.0, 5.0, -5.0], "side_lengths": [14.0, 10.0, 4.0], "min_point": [-2.0, 0.0, -7.0], "max_point": [12.0, 10.0, -3.0]},
    {"shape_type": "cuboid", "center": [-4.0, 0.0, 2.0], "side_lengths": [12.0, 10.0, 8.0], "min_point": [-10.0, -5.0, -2.0], "max_point": [2.0, 5.0, 6.0]},
    {"shape_type": "cuboid", "center": [-3.0, -3.0, -3.0], "side_lengths": [12.0, 12.0, 12.0], "min_point": [-9.0, -9.0, -9.0], "max_point": [3.0, 3.0, 3.0]},
    {"shape_type": "cuboid", "center": [8.0, 2.0, 6.0], "side_lengths": [6.0, 16.0, 8.0], "min_point": [5.0, -6.0, 2.0], "max_point": [11.0, 10.0, 10.0]},
    {"shape_type": "cuboid", "center": [5.0, -4.0, 5.0], "side_lengths": [10.0, 8.0, 8.0], "min_point": [0.0, -8.0, 1.0], "max_point": [10.0, 0.0, 9.0]},
    {"shape_type": "cuboid", "center": [-5.0, 10.0, 3.0], "side_lengths": [8.0, 4.0, 14.0], "min_point": [-9.0, 8.0, -4.0], "max_point": [-1.0, 12.0, 10.0]},
    {"shape_type": "cuboid", "center": [0.0, 0.0, 6.0], "side_lengths": [12.0, 12.0, 12.0], "min_point": [-6.0, -6.0, 0.0], "max_point": [6.0, 6.0, 12.0]},
    {"shape_type": "cuboid", "center": [2.0, -7.0, 5.0], "side_lengths": [20.0, 6.0, 10.0], "min_point": [-8.0, -10.0, 0.0], "max_point": [12.0, -4.0, 10.0]},
    {"shape_type": "cuboid", "center": [-2.0, 3.0, -5.0], "side_lengths": [16.0, 4.0, 12.0], "min_point": [-10.0, 1.0, -11.0], "max_point": [6.0, 5.0, 1.0]},
    {"shape_type": "cuboid", "center": [7.0, 0.0, 6.0], "side_lengths": [12.0, 8.0, 8.0], "min_point": [1.0, -4.0, 2.0], "max_point": [13.0, 4.0, 10.0]},
    {"shape_type": "cuboid", "center": [3.0, 8.0, -4.0], "side_lengths": [10.0, 6.0, 8.0], "min_point": [-2.0, 5.0, -8.0], "max_point": [8.0, 11.0, 0.0]},
    {"shape_type": "cuboid", "center": [-3.0, 3.0, -3.0], "side_lengths": [12.0, 12.0, 6.0], "min_point": [-9.0, -3.0, -6.0], "max_point": [3.0, 9.0, 0.0]},
    {"shape_type": "cuboid", "center": [6.0, -9.0, 1.0], "side_lengths": [14.0, 8.0, 10.0], "min_point": [-1.0, -13.0, -4.0], "max_point": [13.0, -5.0, 6.0]},
    {"shape_type": "cuboid", "center": [0.0, 4.0, -2.0], "side_lengths": [10.0, 8.0, 6.0], "min_point": [-5.0, 0.0, -5.0], "max_point": [5.0, 8.0, 1.0]},
    {"shape_type": "cuboid", "center": [0.0, 5.0, 0.0], "side_lengths": [10.0, 10.0, 10.0], "min_point": [-5.0, 0.0, -5.0], "max_point": [5.0, 10.0, 5.0]},
    {"shape_type": "cuboid", "center": [9.0, -1.0, 4.0], "side_lengths": [6.0, 12.0, 8.0], "min_point": [6.0, -7.0, 0.0], "max_point": [12.0, 5.0, 8.0]},
    {"shape_type": "cuboid", "center": [-7.0, 0.0, 8.0], "side_lengths": [4.0, 20.0, 6.0], "min_point": [-9.0, -10.0, 5.0], "max_point": [-5.0, 10.0, 11.0]},
    {"shape_type": "cuboid", "center": [4.0, 8.0, 3.0], "side_lengths": [8.0, 16.0, 6.0], "min_point": [0.0, 0.0, 0.0], "max_point": [8.0, 16.0, 6.0]},
    {"shape_type": "cuboid", "center": [4.0, -2.0, 7.0], "side_lengths": [10.0, 6.0, 8.0], "min_point": [-1.0, -5.0, 3.0], "max_point": [9.0, 1.0, 11.0]},
    {"shape_type": "cuboid", "center": [-2.0, 5.0, 1.0], "side_lengths": [12.0, 8.0, 8.0], "min_point": [-8.0, 1.0, -3.0], "max_point": [4.0, 9.0, 5.0]},
    {"shape_type": "cuboid", "center": [-1.0, 6.0, 2.0], "side_lengths": [12.0, 8.0, 10.0], "min_point": [-7.0, 2.0, -3.0], "max_point": [5.0, 10.0, 7.0]},
    {"shape_type": "cuboid", "center": [10.0, 0.0, -4.0], "side_lengths": [8.0, 14.0, 6.0], "min_point": [6.0, -7.0, -7.0], "max_point": [14.0, 7.0, -1.0]},
    {"shape_type": "cuboid", "center": [8.0, -6.0, 4.0], "side_lengths": [12.0, 8.0, 8.0], "min_point": [2.0, -10.0, 0.0], "max_point": [14.0, -2.0, 8.0]},
    {"shape_type": "cuboid", "center": [-6.0, -4.0, 9.0], "side_lengths": [6.0, 10.0, 4.0], "min_point": [-9.0, -9.0, 7.0], "max_point": [-3.0, 1.0, 11.0]},
    {"shape_type": "cuboid", "center": [0.0, -5.0, -8.0], "side_lengths": [20.0, 10.0, 6.0], "min_point": [-10.0, -10.0, -11.0], "max_point": [10.0, 0.0, -5.0]},
    {"shape_type": "cuboid", "center": [3.0, 7.0, 5.0], "side_lengths": [12.0, 8.0, 8.0], "min_point": [-3.0, 3.0, 1.0], "max_point": [9.0, 11.0, 9.0]},
    {"shape_type": "cuboid", "center": [7.0, 3.0, -6.0], "side_lengths": [4.0, 8.0, 12.0], "min_point": [5.0, -1.0, -12.0], "max_point": [9.0, 7.0, 0.0]},
    {"shape_type": "cuboid", "center": [-4.0, 8.0, 0.0], "side_lengths": [16.0, 6.0, 10.0], "min_point": [-12.0, 5.0, -5.0], "max_point": [4.0, 11.0, 5.0]},
    {"shape_type": "cuboid", "center": [4.0, 2.0, -5.0], "side_lengths": [12.0, 8.0, 10.0], "min_point": [-2.0, -2.0, -10.0], "max_point": [10.0, 6.0, 0.0]},
    {"shape_type": "cuboid", "center": [5.0, -8.0, 3.0], "side_lengths": [10.0, 4.0, 14.0], "min_point": [0.0, -10.0, -4.0], "max_point": [10.0, -6.0, 10.0]},
    {"shape_type": "cuboid", "center": [8.0, -6.0, 2.0], "side_lengths": [16.0, 12.0, 12.0], "min_point": [0.0, -12.0, -4.0], "max_point": [16.0, 0.0, 8.0]},
    {"shape_type": "cuboid", "center": [-9.0, 2.0, 6.0], "side_lengths": [6.0, 12.0, 8.0], "min_point": [-12.0, -4.0, 2.0], "max_point": [-6.0, 8.0, 10.0]},
    {"shape_type": "cuboid", "center": [3.0, 0.0, -9.0], "side_lengths": [18.0, 8.0, 6.0], "min_point": [-6.0, -4.0, -12.0], "max_point": [12.0, 4.0, -6.0]},
    {"shape_type": "cuboid", "center": [-5.0, 1.0, 6.0], "side_lengths": [12.0, 12.0, 8.0], "min_point": [-11.0, -5.0, 2.0], "max_point": [1.0, 7.0, 10.0]},
    {"shape_type": "cuboid", "center": [6.0, 6.0, -2.0], "side_lengths": [8.0, 8.0, 16.0], "min_point": [2.0, 2.0, -10.0], "max_point": [10.0, 10.0, 6.0]},
    {"shape_type": "cuboid", "center": [2.0, 5.0, -1.0], "side_lengths": [12.0, 10.0, 12.0], "min_point": [-4.0, 0.0, -7.0], "max_point": [8.0, 10.0, 5.0]},
    {"shape_type": "cuboid", "center": [-5.0, -7.0, 4.0], "side_lengths": [14.0, 6.0, 10.0], "min_point": [-12.0, -10.0, -1.0], "max_point": [2.0, -4.0, 9.0]},
    {"shape_type": "cuboid", "center": [1.0, -3.0, 11.0], "side_lengths": [12.0, 10.0, 8.0], "min_point": [-5.0, -8.0, 7.0], "max_point": [7.0, 2.0, 15.0]}
]

## 3. Approach 1: Two-Pass Extraction
Pass 1 generates unstructured math iteratively validating list conversions.

In [7]:
class BarrierDetailsBasic(BaseModel):
    shape_type: str = Field(
        description="Type of shape. MUST be exactly 'cuboid' or 'sphere'."
    )
    center: list[float] = Field(
        description="The center of the object as [x, y, z]"
    )
    side_lengths: list[float] = Field(
        description="Lengths of the sides as [x, y, z]"
    )
    min_point: list[float] = Field(
        description="Minimum bounding point [x, y, z]."
    )
    max_point: list[float] = Field(
        description="Maximum bounding point [x, y, z]."
    )

def extract_two_pass(prompt_text: str, model_name: str = 'llama3.1') -> list:
    model_context = """You are a precise geospatial math assistant. 
1. Shape Resolution: Any square-shaped barrier MUST be strictly labeled as 'cuboid'. Even if the text says 'box' or 'cube', you MUST explicitly identify and output 'cuboid'. If it says 'sphere', output 'sphere'.
2. Extract the center coordinates, and side lengths from the user text. 
3. Calculate the min and max points for the X, Y, and Z axes separately. 
   Formula: min = center - (side / 2) | max = center + (side / 2)."""

    response_1 = chat(
        model=model_name,
        messages=[
            {'role': 'system', 'content': model_context},
            {'role': 'user', 'content': prompt_text}
        ],
        options={'temperature': 0}
    )
    unstructured_math = response_1.message.content

    pass2_system = "You are a data formatting engine. Extract the final calculated parameters from the provided text into the requested structured output schema."
    response_2 = chat(
        model=model_name,
        messages=[
            {'role': 'system', 'content': pass2_system},
            {'role': 'user', 'content': unstructured_math}
        ],
        format=BarrierDetailsBasic.model_json_schema(),
        options={'temperature': 0}
    )
    
    validated = BarrierDetailsBasic.model_validate_json(response_2.message.content)
    return list(validated.model_dump().values())

### Single Test Case - Approach 1

In [10]:
test_id = 30
extracted_list_1 = extract_two_pass(prompts[test_id])
_ = evaluate_test_case(test_id, extracted_list_1)


================ TEST CASE 30 ================
Question:    The box extends from (−2, −2, −10) to (10, 6, 0). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.
--------------------------------------------------
Ollama Ans:  ['cuboid', [4.0, 2.0, -5.0], [12.0, 8.0, 10.0], [-2.0, -2.0, -10.0], [10.0, 6.0, 0.0]]
True Answer: ['cuboid', [4.0, 2.0, -5.0], [12.0, 8.0, 10.0], [-2.0, -2.0, -10.0], [10.0, 6.0, 0.0]]
--------------------------------------------------
Full Match:          ✅
Center/Length Match: ✅
Min/Max Match:       ✅


### Batch Inference - Approach 1

In [8]:
outputs_1 = []
total_cases = len(prompts)

print(f"Starting Approach 1 inference on {total_cases} cases...")
for i in range(total_cases):
    try:
        out1 = extract_two_pass(prompts[i])
        outputs_1.append(out1)
    except Exception as e:
        print(f"Failed on test {i} with exception: {e}")
        outputs_1.append(None)

Starting Approach 1 inference on 40 cases...


### Batch Test Evaluation - Approach 1

In [9]:
correct_1_full = 0
correct_1_cl = 0
correct_1_mm = 0

print(f"Evaluating the {total_cases} Approach 1 outputs...")
for i in range(total_cases):
    if outputs_1[i] is None: 
        continue
        
    act_list = outputs_1[i]
    exp_dict = expected_answers[i]
    
    exp_list = [
        exp_dict.get("shape_type"),
        exp_dict.get("center"),
        exp_dict.get("side_lengths"),
        exp_dict.get("min_point"),
        exp_dict.get("max_point")
    ]
    
    if act_list == exp_list: 
        correct_1_full += 1
    if check_center_length(act_list, exp_dict): 
        correct_1_cl += 1
    if check_min_max(act_list, exp_dict): 
        correct_1_mm += 1

print(f"\n=== Approach 1 (Two-Pass) Final Results ===")
print(f"Full Match:          {correct_1_full}/{total_cases} ({correct_1_full/total_cases*100:.2f}%)")
print(f"Center/Length Match: {correct_1_cl}/{total_cases} ({correct_1_cl/total_cases*100:.2f}%)")
print(f"Min/Max Match:       {correct_1_mm}/{total_cases} ({correct_1_mm/total_cases*100:.2f}%)")

Evaluating the 40 Approach 1 outputs...

=== Approach 1 (Two-Pass) Final Results ===
Full Match:          21/40 (52.50%)
Center/Length Match: 36/40 (90.00%)
Min/Max Match:       21/40 (52.50%)


## 4. Approach 2: Single-Pass Extraction with Explicit Reasoning (Few-Shot Prompt)
Consistently enforces 'cuboid' and outputs exactly the list schema.

In [4]:
class ReasoningMixin:
    reasoning: str = Field(
        ...,
        description="Explain the step-by-step thought process behind the provided values. Include key considerations and how they influenced the final decisions. MUST calculate X, Y, and Z axis independently using center +/- (side / 2).",
        repr=False, exclude=True
    )

class BarrierDetailsReasoning(BaseModel, ReasoningMixin):
    shape_type: str = Field(
        description="Type of shape. MUST be exactly 'cuboid' or 'sphere'."
    )
    center: list[float] = Field(
        description="The center of the object as [x, y, z]"
    )
    side_lengths: list[float] = Field(
        description="Lengths of the sides as [x, y, z]"
    )
    min_point: list[float] = Field(
        description="Minimum bounding point [x, y, z]. From your reasoning, this is [min_x, min_y, min_z]."
    )
    max_point: list[float] = Field(
        description="Maximum bounding point [x, y, z]. From your reasoning, this is [max_x, max_y, max_z]."
    )

def extract_single_pass_reasoning(prompt_text: str, model_name: str = 'llama3.1') -> list:
    system_prompt = (
        'You are a precise geospatial math assistant. \n'
        '1. Shape Resolution: Any square-shaped barrier MUST be strictly labeled as "cuboid". Even if the text says "box" or "cube", you MUST explicitly identify and output "cuboid". If it says "sphere", output "sphere".\n'
        '2. You MUST calculate each axis (X, Y, Z) separately in the reasoning layer. Never mix them. \n'
        'sides 10 x 6 x 4 means side_x=10, side_y=6, side_z=4. \n'
        'Calculate min and max for each axis independently using center - (side / 2) and center + (side / 2). \n\n'
        'EXAMPLE:\n'
        'Text: "A cube centered at (2, 4, 0) with sides 10 x 6 x 4"\n'
        'Reasoning:\n'
        '- Shape resolution: The text says cube, which must be strictly labeled as cuboid.\n'
        '- X axis: center=2, side=10. min_x = 2 - (10/2) = -3. max_x = 2 + (10/2) = 7.\n'
        '- Y axis: center=4, side=6. min_y = 4 - (6/2) = 1. max_y = 4 + (6/2) = 7.\n'
        '- Z axis: center=0, side=4. min_z = 0 - (4/2) = -2. max_z = 0 + (4/2) = 2.\n'
    )
    
    response = chat(
        model=model_name,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': prompt_text}
        ],
        format=BarrierDetailsReasoning.model_json_schema(),
        options={'temperature': 0}
    )
    
    validated = BarrierDetailsReasoning.model_validate_json(response.message.content)
    return list(validated.model_dump().values())

### Single Test Case - Approach 2

In [12]:
test_id = 30
extracted_list_2 = extract_single_pass_reasoning(prompts[test_id])
_ = evaluate_test_case(test_id, extracted_list_2)
# extracted_list_2


================ TEST CASE 30 ================
Question:    The box extends from (−2, −2, −10) to (10, 6, 0). Write the result as the shape (cuboid), centre of the cuboid, lengths of each side (x,y,z), the minimum point and maximum point.
--------------------------------------------------
Ollama Ans:  ['cuboid', [4.0, 2.0, -5.0], [12.0, 8.0, 10.0], [-6.0, -6.0, -10.0], [10.0, 6.0, 0.0]]
True Answer: ['cuboid', [4.0, 2.0, -5.0], [12.0, 8.0, 10.0], [-2.0, -2.0, -10.0], [10.0, 6.0, 0.0]]
--------------------------------------------------
Full Match:          ❌
Center/Length Match: ✅
Min/Max Match:       ❌


### Batch Inference - Approach 2

In [5]:
outputs_2 = []
total_cases = len(prompts)

print(f"Starting Approach 2 inference on {total_cases} cases...")
for i in range(total_cases):
    try:
        out2 = extract_single_pass_reasoning(prompts[i])
        outputs_2.append(out2)
    except Exception as e:
        print(f"Failed on test {i} with exception: {e}")
        outputs_2.append(None)

Starting Approach 2 inference on 40 cases...


### Batch Test Evaluation - Approach 2

In [6]:
correct_2_full = 0
correct_2_cl = 0
correct_2_mm = 0

print(f"Evaluating the {total_cases} Approach 2 outputs...")
for i in range(total_cases):
    if outputs_2[i] is None: 
        continue
        
    act_list = outputs_2[i]
    exp_dict = expected_answers[i]
    
    exp_list = [
        exp_dict.get("shape_type"),
        exp_dict.get("center"),
        exp_dict.get("side_lengths"),
        exp_dict.get("min_point"),
        exp_dict.get("max_point")
    ]
    
    if act_list == exp_list: 
        correct_2_full += 1
    if check_center_length(act_list, exp_dict): 
        correct_2_cl += 1
    if check_min_max(act_list, exp_dict): 
        correct_2_mm += 1

print(f"\n=== Approach 2 (Single-Pass Reasoning) Final Results ===")
print(f"Full Match:          {correct_2_full}/{total_cases} ({correct_2_full/total_cases*100:.2f}%)")
print(f"Center/Length Match: {correct_2_cl}/{total_cases} ({correct_2_cl/total_cases*100:.2f}%)")
print(f"Min/Max Match:       {correct_2_mm}/{total_cases} ({correct_2_mm/total_cases*100:.2f}%)")

Evaluating the 40 Approach 2 outputs...

=== Approach 2 (Single-Pass Reasoning) Final Results ===
Full Match:          6/40 (15.00%)
Center/Length Match: 34/40 (85.00%)
Min/Max Match:       10/40 (25.00%)
